# TNG300: coeval benchmark → finite patch → lightcone correlations

目的は [Zhang & Lidz (arXiv:2512.06584v2)](https://arxiv.org/html/2512.06584v2) のFig. 1・3の条件へ戻り、
現在のlightconeとの違いを一段ずつ調べること。観測beam・雑音・photo-z誤差は使用しない。

1. TNG300-3、snapshot 99（z=0）、箱全体、z軸投影、500² mapでFig. 1に対応する4枚を出す。
2. 論文Fig. 3に対応するcoevalな $w_{R^2,h}(r_\perp;z)$ と $w_{|R|,h}$ を各snapshotで測る。
3. 同じsnapshotでhalo/galaxy tracer、格子へのdeposit方法、全箱/切り出し領域を比較する。
4. `tng_fr_basic_figures.ipynb` の**未平滑化**lightcone出力を読み、端をwrapせず $w(\theta)$ を測る。
5. 対象z層・外側の層・層間交差項へ分解し、lightconeで増える揺らぎを確認する。

論文は具体的なassignment kernelとcatalog selectionの全詳細を公開本文だけでは特定できない。
本notebookは条件を揃える検証用で、数値的に完全な再現を保証するものではない。
coeval格子はvolume-weighted NGPを用い、別々の $n_e,B$ の格子化と直接 $n_eB$ depositを比較する。
全セル・全chunkを読む。500³の一時memory-mapped格子は約3.73 GiB、完成した2D cacheだけを残す。

自己完結したnotebook。numpy/scipy/pandas/matplotlib/h5py/astropyが必要。API key・download不要。
PDF/300 dpi PNGとCSV/NPZを `fr_coeval_to_lightcone/`、中間cacheを `cache/` に保存。

In [ ]:
from pathlib import Path
import json,hashlib,time,warnings
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.signal import fftconvolve
from astropy.cosmology import FlatLambdaCDM
from IPython.display import display

## 1. 設定

通常は `BASE_PATH` のみ確認してRun All。coeval部分は他notebookに依存しない。
lightcone部分のみ既存の `fr_basic_figures/` の出力を使う。
既定snapshotはFig. 3のz≈0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.7, 1, 1.5, 2, 3, 4に対応。
`HALO_CATALOG='auto'` はz=0の全FoF/全subhalo数をFig. 2記載の391,144と照合する。
一方だけ一致した場合はそれを選ぶ。不一致時は本文のsubhalo positionsに従い全subhaloを使い、警告する。
数の一致は選択条件の完全一致の証明ではない。明示的に `'subhalo'` または `'fof'` も指定できる。

In [ ]:
NOTEBOOK_DIR=(Path.cwd()/'detect-mag' if
              (Path.cwd()/'detect-mag'/'tng_fr_coeval_to_lightcone.ipynb').is_file() else Path.cwd())
SIMULATION='TNG300-3'
BASE_PATH=None
CACHE_DIR=NOTEBOOK_DIR/'cache'
OUTPUT_DIR=NOTEBOOK_DIR/'fr_coeval_to_lightcone'
LIGHTCONE_DIR=NOTEBOOK_DIR/'fr_basic_figures'
RUN_LIGHTCONE=True
SNAPSHOTS=[99,91,84,78,72,67,59,50,40,33,25,21]
FIG1_SNAPSHOT=99
ANCHOR_SNAPSHOT=33  # same epoch for tracer/kernel/crop controls, z~2
GRID_N=500         # 205/500 = 0.41 h^-1 cMpc
HALO_CATALOG='auto'
FIG2_HALO_COUNT=391144
FIELD_METHOD='separate'  # 'separate': ne-grid x volume-weighted B; 'direct': deposited Ne B
USE_CELL_XH=True
EXCLUDE_STAR_FORMING=False
READ_BATCH=250_000
N_RADIAL_BINS=28
MAX_R_HMPC=102.5
CONTROL_FOV_DEG=1.5
GALAXY_MIN_STELLAR_MASS=0.0
GALAXY_MIN_STAR_PARTICLES=1
DPI=300
assert GRID_N>=8 and FIELD_METHOD in ('separate','direct')
assert HALO_CATALOG in ('auto','subhalo','fof')
assert FIG1_SNAPSHOT in SNAPSHOTS and ANCHOR_SNAPSHOT in SNAPSHOTS
for p in (CACHE_DIR,OUTPUT_DIR):p.mkdir(parents=True,exist_ok=True)
plt.rcParams.update({'font.size':11,'figure.dpi':110,'savefig.dpi':DPI,'pdf.fonttype':42})
def savefig(fig,name):
    fig.savefig(OUTPUT_DIR/(name+'.pdf'),bbox_inches='tight')
    fig.savefig(OUTPUT_DIR/(name+'.png'),dpi=DPI,bbox_inches='tight');plt.show()
def key_for(meta):return hashlib.sha256(json.dumps(meta,sort_keys=True).encode()).hexdigest()[:20]
def manifest(files):return [(p.name,p.stat().st_size,p.stat().st_mtime_ns) for p in files]
def file_hash(p):
    d=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):d.update(b)
    return d.hexdigest()

## 2. データ・catalog・coeval projection

halo fieldは全FoFまたは全subhaloを使用し、星の有無で絞らない。
galaxy fieldだけは星を持つ有効なsubhaloを使用する。両方のcountを記録する。
各gasセルのcomoving volumeをMasses/Densityから求め、物理磁場をvolume-weightedで格子化する。
電子密度は電子数を格子体積で割る。空の格子の磁場は0とする。

In [ ]:
MSUN_G=1.98847e33;KPC_CM=3.0856775814913673e21;MP_G=1.67262192369e-24
VERSION='coeval_ne_B_NGP_v1'
def files_at(base,snap,kind):
    folder=base/(f'snapdir_{snap:03d}' if kind=='gas' else f'groups_{snap:03d}')
    pattern=f'snap_{snap:03d}.*.hdf5' if kind=='gas' else f'fof_subhalo_tab_{snap:03d}.*.hdf5'
    return sorted(folder.glob(pattern),key=lambda p:int(p.name.split('.')[-2]))
def find_base():
    if BASE_PATH is not None:return Path(BASE_PATH).expanduser().resolve()
    candidates=[]
    provenance=LIGHTCONE_DIR/'figure_configuration.json'
    if provenance.is_file():
        prior=json.loads(provenance.read_text())
        candidates.append(Path(prior.get('theory_cone_metadata',{}).get('base','/missing')))
    for root in [NOTEBOOK_DIR,*list(NOTEBOOK_DIR.parents)[:4],Path.home()]:
        candidates += [root/'sims.TNG'/SIMULATION/'output']
    for p in candidates:
        if (p/'snapdir_099').is_dir():return p.resolve()
    raise FileNotFoundError('Set BASE_PATH to the TNG300-3 output mount.')
def header_for(files,kind):
    if not files:raise FileNotFoundError(f'Missing {kind} files')
    with h5py.File(files[0],'r') as f:hd=dict(f['Header'].attrs)
    expected=int(hd['NumFilesPerSnapshot'] if kind=='gas' else hd['NumFiles'])
    if len(files)!=expected:raise FileNotFoundError(f'Incomplete {kind} chunks')
    return hd
base=find_base()
hd=header_for(files_at(base,99,'gas'),'gas')
h=float(hd['HubbleParam']);box=float(hd['BoxSize'])/(1000*h);box_h=box*h
cosmo=FlatLambdaCDM(H0=100*h,Om0=float(hd['Omega0']))
print('TNG mount:',base,'; box:',box_h,'h^-1 cMpc; voxel:',box_h/GRID_N,'h^-1 cMpc')
if not np.isclose(box_h,205):warnings.warn('Box does not match the paper TNG300 box.')

def read_catalog(snap):
    files=files_at(base,snap,'group');header_for(files,'group')
    fof=[];sub=[];gal=[]
    for path in files:
        with h5py.File(path,'r') as f:
            if 'Group' in f and 'GroupPos' in f['Group']:fof.append(f['Group/GroupPos'][:].astype(float)/(1000*h))
            if 'Subhalo' not in f:continue
            g=f['Subhalo'];pos=g['SubhaloPos'][:].astype(float)/(1000*h);sub.append(pos)
            mass=g['SubhaloMassType'][:,4].astype(float)*1e10/h
            stars=g['SubhaloLenType'][:,4]
            if 'SubhaloFlag' not in g:raise KeyError('SubhaloFlag required for galaxy controls')
            keep=(mass>0)&(mass>=GALAXY_MIN_STELLAR_MASS)&(stars>=GALAXY_MIN_STAR_PARTICLES)&g['SubhaloFlag'][:].astype(bool)
            gal.append(pos[keep])
    combine=lambda parts:np.concatenate(parts) if parts else np.empty((0,3))
    return dict(fof=combine(fof),subhalo=combine(sub),galaxy=combine(gal))
cat0=read_catalog(99)
counts0={k:len(v) for k,v in cat0.items()}
if HALO_CATALOG=='auto':
    matches=[k for k in ['fof','subhalo'] if counts0[k]==FIG2_HALO_COUNT]
    halo_kind=matches[0] if len(matches)==1 else 'subhalo'
    if len(matches)!=1:warnings.warn(f'Paper tracer count not uniquely matched: {counts0}; using all subhalos. Confirm HALO_CATALOG.')
else:halo_kind=HALO_CATALOG
if not counts0[halo_kind]:raise ValueError(f'Empty selected halo catalog: {halo_kind}')
print('z=0 catalog counts:',counts0,'; selected halo catalog:',halo_kind)
del cat0

def counts_map(pos):
    ix=np.floor((pos[:,:2]%box)/box*GRID_N).astype(np.int64)
    return np.bincount(ix[:,0]*GRID_N+ix[:,1],minlength=GRID_N**2).reshape(GRID_N,GRID_N)

def coeval_products(snap):
    gas_files=files_at(base,snap,'gas');groups=files_at(base,snap,'group')
    hd=header_for(gas_files,'gas');header_for(groups,'group')
    gh=float(hd['HubbleParam']);gb=float(hd['BoxSize'])/(1000*gh)
    if not np.isclose(gh,h) or not np.isclose(gb,box):raise ValueError('Snapshot cosmology/box mismatch')
    z=float(hd['Redshift']);a=1/(1+z)
    meta=dict(version=VERSION,base=str(base),simulation=SIMULATION,snap=int(snap),grid=GRID_N,
        halo_kind=halo_kind,cell_XH=USE_CELL_XH,exclude_SF=EXCLUDE_STAR_FORMING,
        galaxy_min_mass=GALAXY_MIN_STELLAR_MASS,galaxy_min_stars=GALAXY_MIN_STAR_PARTICLES,
        gas=manifest(gas_files),groups=manifest(groups),B_unit=2.60e-6)
    path=CACHE_DIR/f'coeval_projection_{key_for(meta)}.npz'
    if path.exists():
        with np.load(path,allow_pickle=False) as f:p={k:f[k].copy() for k in f.files}
        print('Coeval cache:',snap,path.name);return p,meta
    tmp_grid=CACHE_DIR/f'coeval_work_{key_for(meta)}.tmp.npy'
    grid=np.lib.format.open_memmap(tmp_grid,mode='w+',dtype=np.float64,shape=(4,GRID_N,GRID_N,GRID_N));grid[:]=0
    # ne_com, volume_fraction*Bz_phys, volume_fraction, direct ne_com*Bz_phys
    volume=(box/GRID_N*1000*KPC_CM)**3
    used=read_count=0
    for j,path_gas in enumerate(gas_files):
        with h5py.File(path_gas,'r') as f:
            if 'PartType0' not in f:continue
            g=f['PartType0']
            for field in ['Coordinates','Masses','Density','ElectronAbundance','MagneticField']:
                if field not in g:raise KeyError(f'{path_gas}: missing {field}')
            if USE_CELL_XH and 'GFM_Metals' not in g:raise KeyError('Missing GFM_Metals')
            if EXCLUDE_STAR_FORMING and 'StarFormationRate' not in g:raise KeyError('Missing SFR')
            for start in range(0,len(g['Masses']),READ_BATCH):
                sl=slice(start,start+READ_BATCH)
                pos=g['Coordinates'][sl].astype(float)/(1000*h)
                mass_code=g['Masses'][sl].astype(float);rho=g['Density'][sl].astype(float)
                xe=g['ElectronAbundance'][sl].astype(float)
                xh=g['GFM_Metals'][sl,0].astype(float) if USE_CELL_XH else np.full(len(xe),0.76)
                bz=g['MagneticField'][sl,2].astype(float)*(h/a**2)*2.60
                good=np.isfinite(pos).all(1)&np.isfinite(mass_code)&(mass_code>0)&np.isfinite(rho)&(rho>0)
                good &= np.isfinite(xe)&(xe>=0)&np.isfinite(xh)&(xh>0)&(xh<=1)&np.isfinite(bz)
                if EXCLUDE_STAR_FORMING:good &= g['StarFormationRate'][sl]<=0
                read_count+=len(xe);used+=int(good.sum())
                ijk=np.floor((pos[good]%box)/box*GRID_N).astype(np.int64)
                idx=(ijk[:,0]*GRID_N+ijk[:,1])*GRID_N+ijk[:,2]
                ne=mass_code[good]*1e10*MSUN_G/h*xe[good]*xh[good]/MP_G/volume
                vf=mass_code[good]/rho[good]*(KPC_CM/h)**3/volume
                for k,w in enumerate([ne,vf*bz[good],vf,ne*bz[good]]):np.add.at(grid[k].reshape(-1),idx,w)
        print(f'coeval snap {snap}: gas chunks {j+1}/{len(gas_files)}')
    if not used:raise ValueError('No usable gas cells')
    rm_sep=np.zeros((GRID_N,GRID_N));rm_dir=rm_sep.copy();ne_proj=rm_sep.copy();bz_proj=rm_sep.copy()
    for x0 in range(0,GRID_N,8):
        sl=slice(x0,min(x0+8,GRID_N))
        ne=grid[0,sl];vf=grid[2,sl]
        bv=np.divide(grid[1,sl],vf,out=np.zeros_like(vf),where=vf>0)
        ne_proj[sl]=ne.mean(axis=2)
        bz_proj[sl]=bv.mean(axis=2)
        rm_sep[sl]=0.812*(ne*bv).sum(axis=2)*(box/GRID_N*1e6)
        rm_dir[sl]=0.812*grid[3,sl].sum(axis=2)*(box/GRID_N*1e6)
    del ne,vf,bv,grid
    # Only temporary intermediate grids created by this function are removed.
    tmp_grid.unlink()
    catalog=read_catalog(snap)
    p=dict(rm_separate=rm_sep,rm_direct=rm_dir,ne_projected=ne_proj,bz_projected=bz_proj,
        halo_count=counts_map(catalog[halo_kind]),galaxy_count=counts_map(catalog['galaxy']),
        z=np.array(z),snap=np.array(snap),N_halo=np.array(len(catalog[halo_kind])),
        N_galaxy=np.array(len(catalog['galaxy'])),N_fof=np.array(len(catalog['fof'])),
        N_subhalo=np.array(len(catalog['subhalo'])),used_cells=np.array(used),read_cells=np.array(read_count))
    if p['halo_count'].sum()==0 or p['ne_projected'].mean()<=0:raise ValueError('Empty halo/electron field')
    temp=path.with_suffix('.tmp.npz');np.savez_compressed(temp,**p);temp.replace(path)
    return p,meta

def chosen_rm(p):return p['rm_'+FIELD_METHOD]

## 3. 相関関数: 周期箱と有限視野

周期箱では、RM²をhalo中心へstackして全体平均を引く操作をFFTで厳密に計算する。
有限視野ではzero-paddingによる**線形**相関を使い、pixel対数で端を補正する。
各annulusでfieldのhalo-weighted meanから同じmask内のuniform random-center meanを引く。

$$
\widehat w_b=\frac{\sum_{b}D_A}{\sum_{b}D_1}
-\frac{\sum_{b}R_A}{\sum_{b}R_1}.
$$
$D_A$ はtracer位置周囲のfieldの和、$D_1$ は利用可能なtracer–pixel対数。
$R_A,R_1$ はuniform random-centerの期待値をmaskの相関で厳密に計算する（random catalogの乱数は不要）。
有限視野のmean subtractionにはintegral constraintが残る。端補正が宇宙分散を除くわけではない。
表のpair数は独立な測定数ではなく、同じ構造を共有する幾何学的な対数。

In [ ]:
def radial_average(field,tracer,spacing,edges,periodic=False,mask=None):
    a=np.asarray(field,float);d=np.asarray(tracer,float)
    if a.shape!=d.shape or a.ndim!=2 or a.shape[0]!=a.shape[1]:raise ValueError('Square matching maps required')
    if not np.isfinite(a).all() or not np.isfinite(d).all() or np.any(d<0):raise ValueError('Nonfinite field or negative counts')
    m=np.ones_like(a) if mask is None else np.asarray(mask,float)
    if m.shape!=a.shape or np.any((m!=0)&(m!=1)):raise ValueError('Binary mask required')
    d=d*m
    if d.sum()<=0:raise ValueError('No tracers inside mask')
    if periodic:
        corr=lambda x,y:np.fft.ifft2(np.fft.fft2(x)*np.fft.fft2(y).conj()).real
        off=np.fft.fftfreq(len(a))*len(a)
    else:
        corr=lambda x,y:fftconvolve(x,y[::-1,::-1],mode='full')
        off=np.arange(-(len(a)-1),len(a))
    DA=corr(a*m,d);D1=np.maximum(corr(m,d),0)
    RA=corr(a*m,m);R1=np.maximum(corr(m,m),0)
    XX,YY=np.meshgrid(off,off,indexing='ij');radius=np.hypot(XX,YY)*spacing
    rows=[]
    for lo,hi in zip(edges[:-1],edges[1:]):
        take=(radius>=lo)&(radius<hi)&(R1>1e-6)
        dp=D1[take].sum();rp=R1[take].sum()
        value=DA[take].sum()/dp-RA[take].sum()/rp if dp>0 and rp>0 else np.nan
        r=np.sum(radius[take]*R1[take])/rp if rp>0 else (lo+hi)/2
        rows.append(dict(r_lo=lo,r_hi=hi,r=r,w=value,tracer_pixel_pairs=dp,random_pixel_pairs=rp,
                         offsets=int(take.sum()),periodic=periodic))
    return pd.DataFrame(rows)

r_edges=np.geomspace(box_h/GRID_N*0.999,min(MAX_R_HMPC,box_h/2),N_RADIAL_BINS+1)
def coeval_corr(p,field=None,tracer=None,periodic=True,edges=r_edges):
    if field is None:field=chosen_rm(p)**2
    if tracer is None:tracer=p['halo_count']
    return radial_average(field,tracer,box_h/GRID_N,edges,periodic=periodic)
def plot_signed(ax,table,label=None,**kwargs):
    ax.plot(table.r,table.w,label=label,**kwargs)
def signed_scale(ax,values):
    values=np.asarray(values);positive=abs(values[np.isfinite(values)])
    scale=max(float(positive.max())*1e-4,1e-30) if len(positive) else 1e-30
    ax.set_yscale('symlog',linthresh=scale);ax.axhline(0,color='0.65',lw=.6)

## 4. Fig. 1対応: z=0・全箱の4 field

halo/electronは $1+\delta$、磁場は投影後の $|\overline{B}_z|$、RMは**平均を引かない** $R^2$。
磁場とRM²の表示色範囲を5–95 percentileにする。これは表示のみで、相関計算の値をclipしない。
磁場の投影は格子のvolume-weighted physical BのLOS平均。
論文の補間・投影weightの全詳細とは厳密一致を確認できないため、定義をここに固定する。

In [ ]:
products={};metadata={}
products[FIG1_SNAPSHOT],metadata[FIG1_SNAPSHOT]=coeval_products(FIG1_SNAPSHOT)
p=products[FIG1_SNAPSHOT]
fields=[p['halo_count']/p['halo_count'].mean(),p['ne_projected']/p['ne_projected'].mean(),
        abs(p['bz_projected']),chosen_rm(p)**2]
titles=[r'$1+\\delta_h$',r'$1+\\delta_e$',r'$|\\overline{B}_z|$',r'$R^2$']
# Raw strings above use a single TeX backslash after this normalization.
titles=[s.replace('\\\\','\\') for s in titles]
units=['','','microGauss',r'rad$^2$ m$^{-4}$']
fig,axes=plt.subplots(2,2,figsize=(10,9),constrained_layout=True)
for i,(ax,field,title,unit) in enumerate(zip(axes.ravel(),fields,titles,units)):
    positive=field[field>0]
    if not len(positive):raise ValueError(f'No positive field for {title}')
    bounds=np.percentile(field,[5,95]) if i>=2 else np.percentile(positive,[1,99])
    lo=max(float(bounds[0]),float(positive.min()));hi=max(float(bounds[1]),lo*1.01)
    im=ax.imshow(np.ma.masked_less_equal(field,0).T,origin='lower',extent=[0,box_h]*2,
        norm=LogNorm(vmin=lo,vmax=hi),cmap='viridis',rasterized=True)
    ax.set(title=title,xlabel=r'x [h$^{-1}$ cMpc]',ylabel=r'y [h$^{-1}$ cMpc]')
    fig.colorbar(im,ax=ax,label=unit,shrink=.85)
fig.suptitle(f"Fig. 1 conditions: {SIMULATION}, z={float(p['z']):.3f}, {GRID_N}²; tracer={halo_kind}; {FIELD_METHOD}")
savefig(fig,'01_coeval_fig1_fields')
np.savez_compressed(OUTPUT_DIR/'fig1_fields.npz',halo=fields[0],electron=fields[1],B_abs=fields[2],RM2=fields[3],box_h=box_h)

## 5. Fig. 3対応: 各時代の周期箱のstacked correlation

横軸はcomoving $h^{-1}$ Mpc、縦軸は相関関数。$C_L$ ではない。
各snapshotのRMと**同じsnapshotのhalo**を相関させる。他の時代のRMは加えない。
負のbinもsymlogで表示して隠さず保存する。論文のようなlog plotだけでは見えない負の値も保持する。

In [ ]:
correlation_rows=[];coeval_tables={}
fig,axes=plt.subplots(3,2,figsize=(11,13),constrained_layout=True)
for index,snap in enumerate(SNAPSHOTS):
    if snap not in products:products[snap],metadata[snap]=coeval_products(snap)
    p=products[snap];row=min(index//4,2)
    for column,(name,field) in enumerate([('RM2',chosen_rm(p)**2),('absRM',abs(chosen_rm(p)))]):
        table=coeval_corr(p,field=field);coeval_tables[(snap,name)]=table
        tagged=table.assign(snap=snap,z=float(p['z']),statistic=name,tracer=halo_kind,method=FIELD_METHOD)
        correlation_rows.append(tagged)
        plot_signed(axes[row,column],table,label=f"z={float(p['z']):.2f}")
for row in range(3):
    for col in range(2):
        ax=axes[row,col];vals=[t.w.to_numpy() for (s,stat),t in coeval_tables.items()
             if min(SNAPSHOTS.index(s)//4,2)==row and stat==('RM2' if col==0 else 'absRM')]
        signed_scale(ax,np.concatenate(vals));ax.set_xscale('log');ax.legend(fontsize=9)
        ax.set(xlabel=r'$r_\\perp$ [h$^{-1}$ cMpc]'.replace('\\\\','\\'),
               ylabel='w [rad² m⁻⁴]' if col==0 else 'w [rad m⁻²]')
fig.suptitle(f'Fig. 3 conditions: coeval, full periodic box; {halo_kind}; {FIELD_METHOD}')
savefig(fig,'02_coeval_fig3_correlations')
pd.concat(correlation_rows,ignore_index=True).to_csv(OUTPUT_DIR/'coeval_correlations.csv',index=False)
pd.DataFrame([dict(snap=s,z=float(p['z']),N_halo=int(p['N_halo']),N_galaxy=int(p['N_galaxy']),
    N_fof=int(p['N_fof']),N_subhalo=int(p['N_subhalo']),used_cells=int(p['used_cells']),
    read_cells=int(p['read_cells']),rm_rms=float(np.std(chosen_rm(p)))) for s,p in products.items()]).to_csv(OUTPUT_DIR/'snapshot_summary.csv',index=False)

## 6. Control 1: tracerとdeposit methodを一つずつ変える

同じsnapshot・同じ箱でhalo→star-bearing galaxyを比較。
同じhaloを使って、格子上で $n_eB$ を掛ける方法と、セルの $N_eB$ を直接depositする方法も比較する。
この差は観測beamではなく数値的な場の定義・格子へのassignmentによる。

In [ ]:
p=products[ANCHOR_SNAPSHOT]
controls={
 'halo; separate ne/B':coeval_corr(p,field=p['rm_separate']**2),
 'halo; direct Ne B':coeval_corr(p,field=p['rm_direct']**2),
 'galaxy; same RM':coeval_corr(p,tracer=p['galaxy_count'])}
fig,ax=plt.subplots(figsize=(7.4,5),constrained_layout=True)
for label,table in controls.items():plot_signed(ax,table,label=label)
signed_scale(ax,np.concatenate([t.w for t in controls.values()]));ax.set_xscale('log');ax.legend()
ax.set(xlabel=r'Transverse separation [h$^{-1}$ cMpc]',ylabel='w(RM², tracer) [rad² m⁻⁴]',
       title=f"Same full box, z={float(p['z']):.3f}")
savefig(fig,'03_tracer_and_assignment_controls')
pd.concat([t.assign(control=k) for k,t in controls.items()]).to_csv(OUTPUT_DIR/'tracer_assignment_controls.csv',index=False)

## 7. Control 2: 全箱→同じ時代の有限視野

anchor zで1.5°に対応する横幅へ箱中心を切り出す。RMのLOS積分は箱全長のまま。
周期全箱、有限cropの端補正、cropを誤って周期化した場合を比較する。
この段階では時間進化や多層RMを追加しない。cropのpixel数で実際の横幅を決め、記録する。

In [ ]:
p=products[ANCHOR_SNAPSHOT]
chi_anchor=float(cosmo.comoving_distance(float(p['z'])).value)
width=min(box,chi_anchor*np.deg2rad(CONTROL_FOV_DEG))
crop_n=max(8,min(GRID_N,int(round(width/box*GRID_N))))
start=(GRID_N-crop_n)//2;sl=slice(start,start+crop_n)
A=chosen_rm(p)[sl,sl]**2;D=p['galaxy_count'][sl,sl]
crop_edges=r_edges[r_edges<=crop_n*box_h/GRID_N/2]
if len(crop_edges)<3:raise ValueError('Too few crop bins; enlarge control FOV or grid.')
crop_controls={
 'full box; periodic':coeval_corr(p,tracer=p['galaxy_count']),
 'crop; edge corrected':radial_average(A,D,box_h/GRID_N,crop_edges),
 'crop; artificial periodicity':radial_average(A,D,box_h/GRID_N,crop_edges,periodic=True)}
fig,ax=plt.subplots(figsize=(7.4,5),constrained_layout=True)
for label,table in crop_controls.items():plot_signed(ax,table,label=label)
signed_scale(ax,np.concatenate([t.w for t in crop_controls.values()]));ax.set_xscale('log');ax.legend()
ax.set(xlabel=r'Transverse separation [h$^{-1}$ cMpc]',ylabel='w(RM², galaxy) [rad² m⁻⁴]',
       title=f'Window control: crop side {crop_n*box_h/GRID_N:.1f} h⁻¹ cMpc')
savefig(fig,'04_finite_window_control')
pd.concat([t.assign(control=k) for k,t in crop_controls.items()]).to_csv(OUTPUT_DIR/'finite_window_control.csv',index=False)

## 8. Control 3: lightconeの同じz層→全経路

`fr_basic_figures/` の未平滑化RM layers・元の連続位置から作ったgalaxy catalog・設定JSONを使う。
coeval 500格子と既存cone 512格子の違い、LOS深さ、ray geometry、tracer selectionは記録する。
ここは厳密に唯一の条件だけを変える比較ではなく、その違いを確認済みでconeへ進む段階。

lightconeでも同じedge-corrected estimatorで**角度の実空間相関** $w(\theta)$ を測れる。
まず $\theta$ をarcminで表示する。幅広いz binで $r_\perp=\chi\theta$ を単一の距離に換算すると曖昧になるため、
このnotebookではその換算を行わない。狭いz binにすれば横方向の物理距離へ変換できる。

In [ ]:
cone_provenance=None;cone_hashes={};cone_tables=[]
if RUN_LIGHTCONE:
    required=['figure_configuration.json','theory_cone_rm_layers.npz','theory_cone_galaxies.csv']
    missing=[name for name in required if not (LIGHTCONE_DIR/name).is_file()]
    if missing:raise FileNotFoundError(f'{LIGHTCONE_DIR}: missing {missing}; run theoretical basic figures first, or RUN_LIGHTCONE=False.')
    cone_hashes={name:file_hash(LIGHTCONE_DIR/name) for name in required}
    cone_provenance=json.loads((LIGHTCONE_DIR/required[0]).read_text())
    if cone_provenance.get('rm_smoothing_arcmin')!=0:raise ValueError('Only unfiltered theoretical cone supported')
    mc=cone_provenance['main_configuration']
    if mc['SIMULATION']!=SIMULATION or not np.isclose(float(mc['h']),h) or not np.isclose(float(mc['box_mpc']),box):
        raise ValueError('Cone and coeval simulation differ')
    with np.load(LIGHTCONE_DIR/required[1],allow_pickle=False) as f:
        layers=f['rm'].astype(float);cone_edges=f['layer_edges'].astype(float)
    galaxies=pd.read_csv(LIGHTCONE_DIR/required[2])
    target=np.asarray(mc['TRUE_Z_EDGES'],float)
    assert np.array_equal(cone_edges[1:-1],target)
    size=layers.shape[1];fov=float(mc['FOV_DEG'])
    assert layers.shape==(len(cone_edges)-1,size,size) and np.isfinite(layers).all()
    assert ((galaxies[['ix','iy']]>=0)&(galaxies[['ix','iy']]<size)).all().all()
    cone_R=layers.sum(0);angular_pixel=fov*60/size
    angular_edges=np.geomspace(angular_pixel*.999,fov*60/2,N_RADIAL_BINS+1)
    fig,axes=plt.subplots(1,2,figsize=(12,5),constrained_layout=True)
    decomposition=[]
    for i,(lo,hi) in enumerate(zip(target[:-1],target[1:])):
        select=(galaxies.z_cos>=lo)&(galaxies.z_cos<hi)
        pix=galaxies.loc[select,'ix'].to_numpy(dtype=int)*size+galaxies.loc[select,'iy'].to_numpy(dtype=int)
        count=np.bincount(pix,minlength=size**2).reshape(size,size)
        if not count.sum():raise ValueError('Empty cone z bin')
        inside=layers[i+1];outside=cone_R-inside
        terms={'within':inside**2,'outside':outside**2,'cross':2*inside*outside,'total':cone_R**2}
        tables={name:radial_average(field,count,angular_pixel,angular_edges) for name,field in terms.items()}
        # Same mask and counts make the estimator linear in the field: exact closure.
        assert np.allclose(tables['total'].w,tables['within'].w+tables['outside'].w+tables['cross'].w,
                           rtol=1e-8,atol=1e-8*max(1.,np.max(abs(tables['total'].w))),equal_nan=True)
        for name,table in tables.items():
            cone_tables.append(table.assign(z_lo=lo,z_hi=hi,term=name,N_gal=int(count.sum()),unit='arcmin'))
        absolute=radial_average(abs(cone_R),count,angular_pixel,angular_edges)
        cone_tables.append(absolute.assign(z_lo=lo,z_hi=hi,term='absRM',N_gal=int(count.sum()),unit='arcmin'))
        plot_signed(axes[0],tables['total'],label=f'{lo:g}<z<{hi:g}')
        plot_signed(axes[1],absolute,label=f'{lo:g}<z<{hi:g}')
        decomposition.append((lo,hi,tables))
    for col,ax in enumerate(axes):
        values=np.concatenate([t.w for t in cone_tables if
            ((t.term.iloc[0]=='total') if col==0 else (t.term.iloc[0]=='absRM'))])
        signed_scale(ax,values);ax.set_xscale('log');ax.legend()
        ax.set(xlabel='Angular separation [arcmin]',ylabel='w [rad² m⁻⁴]' if col==0 else 'w [rad m⁻²]',
               title='Full-path RM² × galaxy' if col==0 else '|Full-path RM| × galaxy')
    savefig(fig,'05_lightcone_real_space_correlations')
    fig,axes=plt.subplots(len(decomposition),1,figsize=(8,4*len(decomposition)),squeeze=False,constrained_layout=True)
    for ax,(lo,hi,tables) in zip(axes[:,0],decomposition):
        for name,table in tables.items():plot_signed(ax,table,label=name)
        signed_scale(ax,np.concatenate([t.w for t in tables.values()]));ax.set_xscale('log');ax.legend(ncol=4)
        ax.set(xlabel='Angular separation [arcmin]',ylabel='w [rad² m⁻⁴]',title=f'{lo:g}<z<{hi:g}: RM layer decomposition')
    savefig(fig,'06_lightcone_layer_decomposition')
    pd.concat(cone_tables,ignore_index=True).to_csv(OUTPUT_DIR/'lightcone_correlations.csv',index=False)
    print('Cone z bins:',list(zip(target[:-1],target[1:])), '; angular estimator has no periodic wrap.')

## 9. 解釈・保存

periodicなcoeval箱はFig. 3との基準比較、有限cropはwindow/tracer samplingの効果、
lightconeの層分解は外側層と層間項の実現ごとの寄与を示す。
全ガスを使ってもNGP・TNG分解能・稀な高RM領域・有限箱の制約は残る。
異なる格子、catalog種別、cone seed、複数視野で検証し、見た目だけで一致を判断しない。
pair数は独立度数として誤差推定に使わない。ここでは観測S/NやGaussian error barを出さない。

In [ ]:
provenance=dict(version=VERSION,simulation=SIMULATION,base=str(base),grid_n=GRID_N,box_h=box_h,
    halo_catalog=halo_kind,z0_catalog_counts=counts0,paper_z0_count=FIG2_HALO_COUNT,
    paper_count_matched=counts0[halo_kind]==FIG2_HALO_COUNT,field_method=FIELD_METHOD,
    coeval_metadata={str(k):v for k,v in metadata.items()},coeval_r_edges_hMpc=r_edges.tolist(),
    crop_n=crop_n,crop_width_hMpc=crop_n*box_h/GRID_N,control_fov_deg=CONTROL_FOV_DEG,
    observation_beam=False,observation_noise=False,radial_estimator='stack-minus-uniform-random; periodic or zero-padded',
    lightcone_source=str(LIGHTCONE_DIR),lightcone_source_hashes=cone_hashes,
    lightcone_provenance=cone_provenance,limitations='NGP kernel and exact halo selection not fully specified by paper; not guaranteed numerical replication',
    versions={k:__import__(k).__version__ for k in ['numpy','scipy','pandas','matplotlib','h5py','astropy']})
(OUTPUT_DIR/'bridge_configuration.json').write_text(json.dumps(provenance,indent=2),encoding='utf-8')
print('Outputs:',OUTPUT_DIR.resolve())
display(pd.DataFrame([dict(file=p.name,KiB=round(p.stat().st_size/1024,1)) for p in sorted(OUTPUT_DIR.iterdir()) if p.is_file()]))